 # Convolitional Vision Transformer (CvT)

 It is hybrid that combines two powerful AI architectures **CNN and Vision Transformers**.


 A normal Vit does :
> Image → cut into patches → turn patches into tokens → Transformers → classification

The problem is ViT treats the image mostly like a sequence of tokens. It doesnot naturally know that nearby pixels are related or that moving an object slightly shouldnot change what it means.

CvT says:
> "Why not give the transformer some CNNs abilities?"

so CvT combines:

CNN
- local spatial information
- Shared weights
- Downsampling
- hierarchical feature extraction

with

Transformer

- Self attention
- global relationships
- dymamic interactions between tokens

so the idea is :

> **CvT= CNN style image processing + Transformer style attention**

A standard transformer doesnt inherently understand: "this token is next to that token", thats why ViT uses positional embeddings.

CvT makes three major changes:

1. Convolutional token embedding:
instead of simply cutting the image into independent, non overlapping patches, CvT uses a convolution.

```
Image
  ↓
Convolution
  ↓
tokens/features
```

This gives the model local spatial information.

2. Convolutional projections inside attention

Normal transformer attention has:

```
Q = XWq
K = XWk
V = XWv

where Wq, Wk and Wv are linear projections.

CvT instead uses convolution-based projections:

X
 ↓
Convolution
 ↓
Q

X
 ↓
Convolution
 ↓
K

X
 ↓
Convolution
 ↓
V

This allows Q,K and V to incorporate local spatial information.

```
3. Hierarchical stages:
ViT basically keeps the token representation at one resolution.  CvT behaves more like CNN:

```
Stage 1
high resolution
many tokens
small features

      ↓ downsample

Stage 2
medium resolution
fewer tokens
richer features

      ↓ downsample

Stage 3
low resolution
very few tokens
high-level features


This is extremely important.

```

so:

> Early stages: Local /simple features like edge, corners, texture etc

> Later stages=Global/semantic features

CvT doesnot use normal transformer because images have spatial structure.

consider:

Cat, Eye nearby pixels are strongly related.


A Convolution naturally looks at a local neighbourhood:

```
┌───┬───┬───┐
│   │   │   │
├───┼───┼───┤
│   │ ● │   │  ← local neighborhood
├───┼───┼───┤
│   │   │   │
└───┴───┴───┘

```

A convolution thereforre has a local receptive field.

A standard self attention layer, on the other hand can directly compare tokens globally.


so

```

Convolution → local understanding
Attention    → global understanding

CvT combines them.

```



**Receptive Field**: The region of input that neuron can "see". For example, a 3*3 convolution sees:

3*3 neighbourhood

so:

CNN → "Look around me."

where self-attention is more like:

Attention → "Look at everything and decide what matters."



### Convolutional Token Embedding: major component of CvT:

```Python
self.proj=nn.Conv2d(
  in_chans,
  embed_dim,
  kernel_size=patch_size,
  stride=stride,
  padding=padding
)
```

Suppose:

```
Input = 224*224*3

and

kernel_size=7
stride=4
embed_dim=64


The convolution processes the image and produces something like:

56 × 56 × 64  depending on padding

now, we have

56 × 56 × 64


then code does:

x = rearrange(x, "b c h w -> b (h w) c")


this changes:

B × C × H × W

into:

B × (H×W) × C


For eg,

B × 64 × 56 × 56

becomes:

B × 3136 × 64


because:

56 × 56 = 3136


now we have:

3136 tokens
each with 64 features

```

This is called token embedding because each spatial location becomes a token:

```
Feature map

┌───┬───┬───┐
│ T1│ T2│ T3│
├───┼───┼───┤
│ T4│ T5│ T6│
├───┼───┼───┤
│ T7│ T8│ T9│
└───┴───┴───┘


Flatten it:
[T1, T2, T3, T4, T5, T6, T7, T8, T9]


Thats the token sequence used by transformer.

```

**OverLapping Patches**

Important difference from ViT,

ViT often uses:

```
patch_size=16
stride=16
```

so, patches dont overlap.


```
[AAAA][BBBB]
[CCCC][DDDD]
```

CvT can use:

```python
kernel=7
stride=4
```
now patches overlap


```
AAAAAAA
   BBBBBBB
      CCCCCCC
```

This is useful because neighbouring regions share information and helps the model preserves local spatial structure.

**Magic of Stride**:

Stride controls how far the convolution moves;

stride =1 means:

Move one pixel at a time.

> larger stride → Fewer output positions → fewer tokens

Therefore:
> stride perfroms spatial downsampling

This is one reason that CvT is computationally efficient.


**CvT architecture**
```
IMAGE
  │
  ▼
┌─────────────────────┐
│ Stage 1             │
│ Conv Token Embedding│
│ Transformer blocks  │
└─────────────────────┘
  │
  ▼
downsample
  │
  ▼
┌─────────────────────┐
│ Stage 2             │
│ Conv Token Embedding│
│ Transformer blocks  │
└─────────────────────┘
  │
  ▼
downsample
  │
  ▼
┌─────────────────────┐
│ Stage 3             │
│ Conv Token Embedding│
│ Transformer blocks  │
└─────────────────────┘
  │
  ▼
Classifier


```

This representation becomes:

```
high resolution
      ↓
medium resolution
      ↓
low resolution
```

while feature richness increases:

```
simple features
      ↓
intermediate features
      ↓
semantic features

```
- CvT gets one of major advantages of CNNs: Multi scale hierarchical representations

while retaining Transformer attention.


transformer block basically does:

```
Input
  │
  ▼
Normalization
  │
  ▼
Self-Attention
  │
  ▼
Residual connection
  │
  ▼
Normalization
  │
  ▼
MLP / FFN
  │
  ▼
Residual connection
  │
  ▼
Output
```
in CvT, the special part is mainly how Q,K and V are produced.


**Normal Transfer Attention**

let X is our token representaion, standard attention computes:

```
Q = X @ Wq
K = X @ Wk
V = X @ Wv
```

Then:

```
Attention(Q,K,V)
=
softmax(QKᵀ / √d)V

```

The important part is:

```
QKᵀ
```
which tells how strongly every token should interact with every other token.

## CvT Attention:

It changes the projection step:

instead of :

```

X → Linear → Q
X → Linear → K
X → Linear → V

it uses convoluional projections:


X
 │
 ├── convolution → Q
 │
 ├── convolution → K
 │
 └── convolution → V

```

Q\K\V contain local spatial information. so attention receives better structured visual features.


### Depthwise Convolution:

```Python
nn.Conv2d(
    dim_in,
    dim_in,
    kernel_size=kernel_size,
    ...
    groups=dim_in
)

```

The key is
```
groups=dim_in
```


this creates a depthwise convolution.

Normally, a convolution mixes information accross channels. Depthwise convolution instead applies a seperate spatial filter to each channel.

let:
```
Channel 1 → filter 1
Channel 2 → filter 2
Channel 3 → filter 3
```

Which is cheaper than a full convolution, which is one reason CvT can maintain efficiency.

### Why remove positional embeddings ?

This is one of the most interesting ideas in CvT:

Vit needs:
token + positional embedding

because otherwise the transformer doesnot inherently know:

Token A is above Token B

But CvT uses convolution.

Convolution naturally operates on grid:

```
┌───┬───┬───┐
│ A │ B │ C │
├───┼───┼───┤
│ D │ E │ F │
├───┼───┼───┤
│ G │ H │ I │
└───┴───┴───┘

The convolution knows that:

E has neighbouring locations:


A B C
D E F
G H I

Therefore, convolution introduces spatial structure.
```

CvT says:

> I dont need a seperate positional embedding because convolution already provides useful positional/local information


### Classification Token

A standard ViT often uses:
[CLS] token at the beginning:

```
[CLS] T1 T2 T3 T4 ...
```
The transformer updates the CLS token through attention.

At the end:

```
CLS → Classifier

```

CvT does something slightly different.

The classification token is only introduced in final stage because,  early stages are primarily processing spatial visual information.

There is no need to carry a special classification token through all early stages.

so conceptually,

```
Stage 1
T1 T2 T3 T4 ...

Stage 2
T1 T2 T3 ...

Stage 3
[CLS] T1 T2 ...

        ↓

      CLS

        ↓

   Linear layer

        ↓

   class prediction
```

### what happens if there is no CLS token ?

The implementation has another options:

instead of:

```
CLS → classifier

```
it can take all final tokens.

```
T1
T2
T3
...
TN

```
and average them:

```python
x=torch.mean(x,dim=1)
```

This is called global average pooling over the token dimensions.

so:

```
N tokens × D features

becomes,

1 × D features

then,

D features
 ↓
Linear
 ↓
classes



```


## Complete Pipeline


```
                 IMAGE
                   │
                   ▼
        Convolutional Token
             Embedding
                   │
                   ▼
            Transformer
              Blocks
                   │
                   ▼
             STAGE 1
                   │
                   │ downsample
                   ▼
        Convolutional Token
             Embedding
                   │
                   ▼
            Transformer
              Blocks
                   │
                   ▼
             STAGE 2
                   │
                   │ downsample
                   ▼
        Convolutional Token
             Embedding
                   │
                   ▼
            Transformer
              Blocks
                   │
                   ▼
             STAGE 3
                   │
                   ▼
         CLS / Global Average
              Pooling
                   │
                   ▼
             LayerNorm
                   │
                   ▼
             Linear Head
                   │
                   ▼
            CLASSIFICATION
```


### ConvolutionalVisionTransformer

```Python
for i in range(self.num_stages):
```
Creates multiple stages

For example:

```
i = 0 → stage0
i = 1 → stage1
i = 2 → stage2
```
Each stage gets its own configuration:

```Python
spec["PATCH_SIZE"][i]
spec["PATCH_STRIDE"][i]
spec["DIM_EMBED"][i]
spec["DEPTH"][i]
spec["NUM_HEADS"][i]

# so each stage can be different

```
For example:

```
Stage 1:
patch = 7
stride = 4
embedding = 64
heads = 1

Stage 2:
patch = 3
stride = 2
embedding = 192
heads = 3

Stage 3:
patch = 3
stride = 2
embedding = 384
heads = 6
```

The exact configuration depends on the CvT variant.


## why does in_chans change?

```python
in_chans = spec["DIM_EMBED"][i]
```

```
Image
3 channels
   ↓
Stage 1
64 channels
   ↓
Stage 2
192 channels
   ↓
Stage 3
384 channels

```

The model is progressively creating richer feature representations.

## Spatial dimensions decreases because the stages use strides:


```

224 × 224
      ↓
56 × 56
      ↓
28 × 28
      ↓
14 × 14


at the same time:

3 channels
 ↓
64
 ↓
192
 ↓
384


so the model trades spatial resolution for feature richness.

```
Which is similar hierarchy to CNNs.

### FLOPs - Why downsampling help ?

Self attention can become expensive when the number of tokens gets large.

if we have N tokens, the attention is roughly,

N × N


so attention has approximately quadriatic dependence on token count.

For example:

```
100 tokens → 10,000 pairwise interactions
1000 tokens → 1,000,000 interactions
```

Thats huge difference. Therefore reducing the number of tokens as we move through stages is valuable.


```
CNN
= excellent local structure

ViT
= excellent global relationships

CvT
= local structure + global relationships
```


-----

1. Why does CvT use Convolution ?

> Because convolution provides local spatial information and spatial structure, which is natural for images.

2. Why does CvT not need traditional postional embeddings ?

> Because convolution operates on spatial neighborhoods and therefore introduces local spatial structure into the representation

3. What is difference between ViT and CvT?

> ViT primarily uses patch embeddingds and standard transformer blocks. CvT adds convolutional token embedding, convolutional Q/K/V projections, and a hierarchical multi stage structure


4. Why does CvT have multiple stages ?

> To progressively reduce spatial resolution while increasing feature richness, similar to a CNN.

5. Why use stride ?
> Stride reduces spatial resolution, which reduces the number of tokens and therefore makes later attention operations cheaper.

6. What are Q, K, and V ?

> They are theree representation used by self attention:

```

Q = What am I looking for?
K = What information do I contain?
V = What information should I pass along?

```

CvT generates them using convolutional projections rather than only linear projections.




> *CvT is a Vision Transformer that incorporates convolution into the Transformer architecture. It uses convolutional token embeddings to capture local spatial information and convolutional projections for Q, K and V inside attention. Unlike standard ViT, CvT is hierarchical: it progressively reduces spatial resolution and increases feature dimensions across multiple stages. Because convolution provides spatial structure, CvT can avoid explicit positional embeddings. The result combines CNN-like local and hierarchical feature extraction with Transformer-like global self-attention.*



```
                 CvT
                  │
       ┌──────────┴──────────┐
       ↓                     ↓
   Convolution           Transformer
       │                     │
 local structure       global attention
       │                     │
       └──────────┬──────────┘
                  ↓
        Hierarchical features
                  ↓
            Classification

```


-----

## 🧠 CNN vs. ViT vs. CvT: Vision Model Selection Guide

---

### 1. In One Sentence

* **CNN:** 👀 *"Look at nearby pixels and gradually build up an understanding."*
* **ViT:** 🌎 *"Look at all image patches and learn how they relate globally."*
* **CvT:** 🧠 *"Use CNNs for local structure, then apply attention for global relationships."*

---

### 2. Quick Model Comparison

| Model | Best When... | Strengths | Weaknesses | Data Requirement |
| --- | --- | --- | --- | --- |
| **CNN** | Efficient image model needed, especially with limited data | Excellent local feature extraction, efficient, mature, fast | Global relationships are less direct | 🟢 Low–Medium |
| **ViT** | Lots of training data/compute & strong global modeling needed | Global self-attention, excellent scalability | Expensive with tokens; needs positional encoding; fewer inductive biases | 🔴 High |
| **CvT** | Want Transformer attention with CNN inductive biases | Local + global info, hierarchical, no traditional positional encoding | More complex than CNN; has attention cost | 🟡 Medium–High |
| **Swin Transformer** | Transformer performance needed for dense vision tasks | Hierarchical, local/window attention, efficient at high resolutions | Complicated architecture (shifted-window mechanism) | 🟡 Medium–High |
| **ResNet** | You need a reliable, robust CNN baseline | Well understood, easy to fine-tune | Older architecture; weaker global modeling | 🟢 Low–Medium |
| **EfficientNet** | High accuracy-to-compute efficiency is critical | Excellent parameter/FLOP efficiency | Less flexible than newer architectures | 🟢 Low–Medium |
| **ConvNeXt** | CNN simplicity meets modern Transformer design | Strong performance, simple architecture, scales well | Lacks self-attention | 🟡 Medium |
| **Hybrid (CNN + ViT)** | Combining local features with global reasoning | Best of both worlds conceptually | More complex and potentially expensive | 🟡–🔴 Medium–High |

---

### 3. The Simplest Decision Tree

```text
START
 │
 ▼
Do you need vision?
 │
 ▼
How much data?
 ├── Small ──► CNN
 └── Large
      │
      ▼
   Need global relationships?
    ├── No  ──► CNN
    └── Yes ──► ViT / CvT
         │
         ├─► Want CNN-like inductive bias? ──► CvT
         └─► Want pure Transformer? ─────────► ViT

```

---

### 4. When to Choose Each Architecture

#### 🟢 Choose CNN when:

* **Context:** You have a small dataset, limited GPU/compute, need fast edge/mobile inference, or want a standard classification/detection task.
* **Example:** *"I have 20,000 labeled manufacturing images and need a fast defect classifier."* Start with **ResNet**, **EfficientNet**, or **ConvNeXt**.

#### 🔵 Choose ViT when:

* **Context:** You have massive datasets, significant compute, and need powerful global relationships for multi-task vision models or foundation models.
* **Example:** *"I'm building a large vision foundation model with millions of training images."*

#### 🟣 Choose CvT when:

* **Context:** You want the dynamic global attention of a Transformer, but also the spatial structure and hierarchical features of a CNN (no traditional positional encoding required).
* **Why it works:** CvT bridges the gap by combining local convolutional info, spatial hierarchy, and global attention.

---

### 5. Task Suitability Matrix

| Task | CNN | ViT | CvT | Swin |
| --- | --- | --- | --- | --- |
| **Image Classification** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Object Detection** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Semantic Segmentation** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Instance Segmentation** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Edge / Mobile** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Huge-Scale Pretraining** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Small Dataset** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |

> *Note: Stars are rough rules of thumb, not strict benchmarks.*

---

### 6. Interview Quick-Reference

> **Q: "Why would you use CvT instead of ViT?"**
> * **Answer:** ViT treats an image as a flat sequence of patches using standard self-attention. **CvT** introduces convolutional token embeddings and projections, giving the model stronger local spatial inductive biases. It is also **hierarchical**, progressively reducing spatial resolution like a CNN, making it ideal for multi-scale vision tasks.
>
>

> **Q: "Why not just use a standard CNN?"**
> * **Answer:** CNNs excel at local feature extraction, but self-attention offers more flexible global interactions. CvT merges convolution's local spatial bias with Transformer's dynamic global attention.
>
>

---

### 7. 🧠 The Feynman Memory Trick

* **CNN:**
> `"What's around me?"` $\rightarrow$ Local $\rightarrow$ Local $\rightarrow$ Global understanding.


* **ViT:**
> `"Which parts of the image are related?"` $\rightarrow$ Global attention from the beginning.


* **CvT:**
> `"What's around me?"* (Convolution) + *"What is related?"* (Attention) $\rightarrow$ Progressive abstraction (Hierarchy).



---

### 8. Final Cheat Sheet

* **Small dataset + limited compute:** CNN
* **Fast / edge / mobile inference:** CNN
* **Reliable baseline:** ResNet
* **Excellent efficiency:** EfficientNet / ConvNeXt
* **Huge dataset + huge compute:** ViT
* **Want CNN inductive bias + Transformer:** CvT
* **Hierarchical Transformer features:** CvT / Swin
* **Building a vision foundation model:** ViT-family

In [ ]:
from transformers import AutoImageProcessor, CvtForImageClassification
from PIL import Image
import requests

# Load image from URL
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

# Use AutoImageProcessor instead of AutoFeatureExtractor
image_processor = AutoImageProcessor.from_pretrained("microsoft/cvt-13")
model = CvtForImageClassification.from_pretrained("microsoft/cvt-13")

# Process the image and run inference
inputs = image_processor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits

print(logits.shape)


predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])

Loading weights:   0%|          | 0/459 [00:00<?, ?it/s]

torch.Size([1, 1000])
Predicted class: tabby, tabby cat
